# Modeling

In this section, we extend our preprocessing pipeline to train and evaluate classification algorithms. Our goal is to identify the best-performing model for predicting customer churn.

## Imports and splitting data

Firstly, we will use our data preprocessing function and split our data for training and test part. The important thing in splitting will be `stratify=y`. This parameter will ensure that in training and test set there will be the same propotion of each class.

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


from utils import split_columns, data_preprocessing

X, y, pre_pipeline = data_preprocessing()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Model Selection and Hyperparameter Tuning

In this experiment, we evaluate three distinct classifiers: **Support Vector Machine (SVM)**, **Decision Tree**, and **Random Forest**. All models are initialized with `class_weight='balanced'` to account for any imbalances in the target classes.

We utilize **Grid Search** to explore the following hyperparameter spaces:

### 1. Support Vector Machine (SVM)
The SVM seeks a hyperplane that maximizes the margin between classes. It is effective for high-dimensional data.
* **`C`** `[0.1, 1, 10]`: The regularization parameter. A smaller $C$ encourages a larger margin (preventing overfitting).
* **`kernel`** `['linear', 'rbf']`: Determines the decision boundary shape. We test a **Linear** kernel for separable data and **RBF** (Radial Basis Function) for non-linear relationships.

### 2. Decision Tree Classifier
A non-parametric model that learns simple decision rules from data features. It provides high interpretability.
* **`max_depth`** `[5, 10, 20]`: Controls the maximum depth of the tree to prevent overfitting (acting as regularization). 
* **`min_samples_split`** `[2, 10, 20]`: The minimum samples required to split an internal node. Higher values constrain the model from learning overly specific rules.
* **`criterion`** `['gini', 'entropy']`: The function to measure the quality of a split. We compare **Gini Impurity** against **Information Gain (Entropy)**.

### 3. Random Forest Classifier
An ensemble method that builds multiple decision trees and merges them to get a more accurate and stable prediction.
* **`n_estimators`** `[50, 100, 150]`: The number of trees in the forest. More trees generally improve stability at the cost of computation.
* **`max_depth`** `[10, 20, 30]`: Limits the depth of individual trees to control model complexity.
* **`min_samples_leaf`** `[1, 4]`: The minimum number of samples required to be at a leaf node. Higher values smooth the model and reduce variance.

In [5]:
models_config = [
        {
            'name': 'Support Vector Machine (SVM)',
            'model': SVC(probability=True, random_state=42, class_weight='balanced'),
            'params': {
                'classifier__C': [0.1, 1, 10],
                'classifier__kernel': ['linear', 'rbf']
            }
        },
        {
            'name': 'Decision Tree',
            'model': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
            'params': {
                'classifier__max_depth': [5, 10, 20],
                'classifier__min_samples_split': [2, 10, 20],
                'classifier__criterion': ['gini', 'entropy']
            }
        },
        {
            'name': 'Random Forest',
            'model': RandomForestClassifier(random_state=42, class_weight='balanced'),
            'params': {
                'classifier__n_estimators': [50, 100, 150],
                'classifier__max_depth': [10, 20, 30],
                'classifier__min_samples_leaf': [1, 4]
            }
        }
    ]


## Model Training and Evaluation Loop

This code iterates through the defined `models_config` to automate the training and tuning process. For each model:

1.  **Pipeline Integration**: The specific classifier is appended to the existing preprocessing steps (`pre_pipeline`).
2.  **Grid Search**: We employ `GridSearchCV` with **5-fold cross-validation** to find the optimal hyperparameters. The search optimizes for the **F1-score**, ensuring a balance between precision and recall, which better than **Accuracy** for our imbalanced dataset.
3.  **Testing & Metrics**: The best estimator found is immediately evaluated on the hold-out test set (`X_test`). We capture **Accuracy, Recall, Precision, and F1-Score** to generate a comprehensive performance comparison.

In [ ]:
results_data = []



for config in models_config:
    print(f"--- TRAINING: {config['name']} ---")

    full_pipeline = Pipeline(steps=pre_pipeline.steps + [('classifier', config['model'])])
    
    grid_search = GridSearchCV(
        estimator=full_pipeline,
        param_grid=config['params'],
        cv=5,              
        scoring='f1', 
        n_jobs=-1,          
        verbose=1
    )
    

    grid_search.fit(X_train, y_train)
    

    best_model = grid_search.best_estimator_
    

    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1] 
    

    acc = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    

    results_data.append({
        'Model': config['name'],
        'Best Params': grid_search.best_params_,
        'Accuracy': acc,
        'Recall' : recall,
        'Precision': precision,
        'F1-Score': f1
    })

    print(f"Best params: {grid_search.best_params_}")
    print(f"F1 on test set: {f1}\n")



    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Stay', 'Exit'])
    disp.plot(cmap='Blues', values_format='d', ax=plt.gca())
    plt.title(f"Confusion Matrix: {config['name']}")
    filename = f"cm_{config['name']}.png"
    plt.savefig(filename) 
    plt.close() 


           Model  Accuracy  F1-Score   ROC-AUC
0  Random Forest    0.8685   0.59098  0.858062


In [ ]:
results_df = pd.DataFrame(results_data).sort_values(by='F1-Score', ascending=False)
results_df.to_csv("model_results.csv", index=False)
print("Results saved in: model_results.csv")

print("="*60)
print("--- MODEL PERFORMANCE SUMMARY ---")
print("="*60)

print(results_df[['Model', 'Accuracy', 'F1-Score', 'Precision', 'Recall']])